# Topic: OWASP Top 10 Defense - Broken Object Level Authorization (BOLA/IDOR) and ownership verification
 
## 1. THE OWASP TOP 10 LIST
- **OWASP Top 10**: A standard awareness document mapping the 10 most critical security risks for web applications (e.g. Broken Access Control, Cryptographic Failures, Injection).
 
## 2. BROKEN OBJECT LEVEL AUTHORIZATION (BOLA / IDOR)
- **Insecure Direct Object Reference (IDOR)**:
  - Occurs when an application provides direct access to objects based on user-supplied inputs (like ID numbers: `/api/invoice/4582`).
  - If the server fails to validate whether the logged-in user actually owns or is authorized to access invoice `4582`, an attacker can swap the ID to view other users' private documents (Horizontal Privilege Escalation).
- **Mitigation (Ownership Verification)**:
  - Never assume user permission based on session validity alone.
  - For every resource request, look up the resource owner ID in the database and verify that it matches the session user ID:
    `assert resource.owner_id == session.user_id`
 
---


In [ ]:
# Mock database records
invoices_db = {
    101: {"owner": "alice", "amount": 150.0, "details": "Alice server subscription"},
    102: {"owner": "alice", "amount": 45.0,  "details": "Alice domain registration"},
    201: {"owner": "bob",   "amount": 2500.0,"details": "Bob private medical consultation"}
}



In [ ]:
# 1. Vulnerable API Endpoint (BOLA/IDOR)
class VulnerableInvoiceAPI:
    """Vulnerable endpoint that only checks if user is authenticated (logged in)."""
    @classmethod
    def get_invoice(cls, user_session_name, invoice_id):
        # 1. Check if user is logged in
        if not user_session_name:
            return "401 Unauthorized"
            
        # 2. Fetch invoice directly without checking ownership (BOLA!)
        invoice = invoices_db.get(invoice_id)
        if not invoice:
            return "404 Not Found"
            
        return invoice

print("--- Vulnerable API IDOR Access ---")
# Alice logs in and reads her own invoice (101) -> Valid usage
print(f"Alice reads invoice 101: {VulnerableInvoiceAPI.get_invoice('alice', 101)}")

# Alice tries to read Bob's invoice (201) -> Vulnerability exploited!
print(f"Alice reads invoice 201: {VulnerableInvoiceAPI.get_invoice('alice', 201)}")
# Expected: Alice successfully views Bob's private database invoice!



In [ ]:
# 2. Secure API Endpoint (Ownership Validation)
class SecureInvoiceAPI:
    """Secure endpoint verifying resource ownership."""
    @classmethod
    def get_invoice(cls, user_session_name, invoice_id):
        # 1. Authenticate user
        if not user_session_name:
            return "401 Unauthorized"
            
        # 2. Fetch invoice
        invoice = invoices_db.get(invoice_id)
        if not invoice:
            return "404 Not Found"
            
        # 3. Secure validation: Verify owner matches active session name
        if invoice["owner"] != user_session_name:
            # Return generic 403 Forbidden (or 404 to prevent resource existence enumeration)
            return "403 Forbidden: Access Denied! You do not own this resource."
            
        return invoice

print("\n--- Secure API IDOR Mitigation ---")
# Alice reads her own invoice (101) -> Allowed
print(f"Alice reads invoice 101: {SecureInvoiceAPI.get_invoice('alice', 101)}")

# Alice tries to read Bob's invoice (201) -> Blocked!
print(f"Alice reads invoice 201: {SecureInvoiceAPI.get_invoice('alice', 201)}")
# Expected: Blocks Alice and returns 403 Forbidden!
